# TMDB pull

Pulls every film with at least 100 TMDB votes, with its reviews in the same request.

Reads `TMDB_API_KEY` from a `.env` file in this folder. Writes `data/tmdb_movies.csv`
and `data/tmdb_reviews.csv`.

The run is resumable: every finished film is appended to `data/pull.jsonl`, and rerunning the
notebook skips films already on disk.

## Setup

In [13]:
import json
import os
import re
import time

import numpy as np
import pandas as pd
import requests
import dotenv

dotenv.load_dotenv()
api_key = os.getenv("TMDB_API_KEY")
len(api_key)

32

In [14]:
BASE = "https://api.themoviedb.org/3"
VOTE_FLOOR = 100
YEARS = range(1874, 2027)
APPEND = "credits,keywords,external_ids,release_dates,reviews,alternative_titles"
RAW = "data/pull.jsonl"
os.makedirs("data", exist_ok=True)

## Enumerate the population

`discover` never returns more than 20,001 films for one query, and the whole >=100-vote
population is 23,708, so the list has to be collected a year at a time. Paging a single
global query would silently drop about 3,700 films.

In [15]:
found = []

for year in YEARS:
    page = 1
    while True:
        r = requests.get(BASE + "/discover/movie", headers={}, params={
            "api_key": api_key,
            "vote_count.gte": VOTE_FLOOR,
            "primary_release_year": year,
            "page": page,
            "sort_by": "vote_count.desc",
        })
        body = r.json()
        for movie in body.get("results", []):
            found.append({
                "tmdb_id": movie["id"],
                "title": movie["title"],
                "release_date": movie["release_date"],
                "vote_count": movie["vote_count"],
            })
        if page >= body.get("total_pages", 1) or page >= 500:
            break
        page += 1
        time.sleep(0.02)
    time.sleep(0.02)

population = pd.DataFrame(found).drop_duplicates("tmdb_id")
population.to_csv("data/tmdb_population.csv", index=False)
len(population)

23690

In [16]:
population.groupby(population.release_date.str[:4]).size().describe().round(1)

count    134.0
mean     176.8
std      249.8
min        1.0
25%       11.2
50%       62.5
75%      218.5
max      978.0
dtype: float64

## Pull each film, with reviews attached

One request per film returns the movie record and its first page of reviews together. Films
already present in `pull.jsonl` are skipped, so an interrupted run picks up where it stopped.

In [17]:
done = set()

if os.path.exists(RAW):
    with open(RAW, encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                done.add(json.loads(line)["tmdb_id"])

todo = population[~population.tmdb_id.isin(done)]
len(done), len(todo)

(23647, 47)

In [18]:
failed = []

with open(RAW, "a", encoding="utf-8") as handle:
    for number, tmdb_id in enumerate(todo.tmdb_id, start=1):
        d = None
        reason = None
        for attempt in range(4):
            try:
                r = requests.get(BASE + f"/movie/{tmdb_id}",
                                 params={"api_key": api_key, "append_to_response": APPEND},
                                 timeout=30)
                if r.status_code == 200:
                    d = r.json()
                    break
                reason = r.status_code
                if r.status_code in (429, 500, 502, 503, 504):
                    time.sleep(2 + attempt * 3)
                    continue
                break
            except requests.RequestException as exc:
                reason = type(exc).__name__
                time.sleep(1 + attempt * 2)
        if d is None:
            failed.append((tmdb_id, reason))
            continue
        crew = d.get("credits", {}).get("crew", [])
        directors = [c["name"] for c in crew if c.get("job") == "Director"]
        writers = [c["name"] for c in crew if c.get("job") in ("Writer", "Screenplay")]
        certification = next(
            (rd["certification"]
             for entry in (d.get("release_dates") or {}).get("results", [])
             if entry.get("iso_3166_1") == "US"
             for rd in entry.get("release_dates", [])
             if rd.get("type") == 3 and rd.get("certification")),
            None)
        alt = (d.get("alternative_titles") or {}).get("titles", [])
        record = {
            "tmdb_id": tmdb_id,
            "imdb_id": (d.get("external_ids") or {}).get("imdb_id"),
            "title": d.get("title"),
            "original_title": d.get("original_title"),
            "release_date": d.get("release_date"),
            "runtime": d.get("runtime"),
            "budget": d.get("budget"),
            "revenue": d.get("revenue"),
            "vote_average": d.get("vote_average"),
            "vote_count": d.get("vote_count"),
            "original_language": d.get("original_language"),
            "genres": "|".join(g["name"] for g in d.get("genres", [])),
            "companies": "|".join(c["name"] for c in d.get("production_companies", [])),
            "countries": "|".join(c["name"] for c in d.get("production_countries", [])),
            "collection": (d.get("belongs_to_collection") or {}).get("name"),
            "cast_count": len(d.get("credits", {}).get("cast", [])),
            "crew_count": len(crew),
            "vfx_crew": sum(1 for c in crew if c.get("department") == "Visual Effects"),
            "director": directors[0] if directors else None,
            "n_directors": len(directors),
            "writers": len(writers),
            "certification": certification,
            "status": d.get("status"),
            "popularity": d.get("popularity"),
            "spoken_languages": "|".join(l["english_name"] for l in d.get("spoken_languages", [])),
            "origin_country": "|".join(d.get("origin_country", [])),
            "keywords": "|".join(k["name"] for k in (d.get("keywords") or {}).get("keywords", [])),
            "alternative_titles": "|".join(t["title"] for t in alt),
            "alt_titles_n": len(alt),
            "review_total": (d.get("reviews") or {}).get("total_results", 0),
            "reviews": [
                {
                    "review_id": rv.get("id"),
                    "author": rv.get("author"),
                    "rating": (rv.get("author_details") or {}).get("rating"),
                    "created_at": rv.get("created_at"),
                    "content": rv.get("content"),
                }
                for rv in (d.get("reviews") or {}).get("results", [])
            ],
        }
        handle.write(json.dumps(record) + "\n")
        if number % 250 == 0:
            handle.flush()
            print(f"{number} of {len(todo)}, {len(failed)} failed")
        time.sleep(0.03)

print(f"finished: {len(todo) - len(failed)} written, {len(failed)} failed")
print(failed[:10])

finished: 47 written, 0 failed
[]


## Flatten to the two tables

Films with more than twenty reviews only returned the first page, so those are topped up
here, one request per page beyond the first.

In [19]:
records = []

with open(RAW, encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            records.append(json.loads(line))

movies = pd.DataFrame([{k: v for k, v in rec.items() if k != "reviews"} for rec in records])
movies.shape

(23694, 30)

In [20]:
short = movies[movies.review_total > 20].tmdb_id.tolist()
len(short)

29

In [21]:
extra = {}

for number, tmdb_id in enumerate(short, start=1):
    pages = []
    page = 2
    while True:
        r = requests.get(BASE + f"/movie/{tmdb_id}/reviews",
                         params={"api_key": api_key, "page": page})
        body = r.json()
        for rv in body.get("results", []):
            pages.append({
                "review_id": rv.get("id"),
                "author": rv.get("author"),
                "rating": (rv.get("author_details") or {}).get("rating"),
                "created_at": rv.get("created_at"),
                "content": rv.get("content"),
            })
        if page >= body.get("total_pages", 1) or page >= 50:
            break
        page += 1
        time.sleep(0.03)
    extra[tmdb_id] = pages
    time.sleep(0.03)

len(extra)

29

In [22]:
rows = []

for rec in records:
    for rv in rec["reviews"] + extra.get(rec["tmdb_id"], []):
        rows.append({"tmdb_id": rec["tmdb_id"], "title": rec["title"], **rv})

reviews = pd.DataFrame(rows).drop_duplicates("review_id")
reviews.shape

(27191, 7)

## Save

In [23]:
movies.to_csv("data/tmdb_movies.csv", index=False)
reviews.to_csv("data/tmdb_reviews.csv", index=False)
movies.shape, reviews.shape

((23694, 30), (27191, 7))

In [24]:
pd.Series({
    "films": len(movies),
    "films with a review": int((movies.review_total > 0).sum()),
    "reviews": len(reviews),
    "films with budget": int((movies.budget > 0).sum()),
    "films with revenue": int((movies.revenue > 0).sum()),
})

films                  23694
films with a review    11509
reviews                27191
films with budget      11254
films with revenue     12710
dtype: int64